In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## imports

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [ ]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [ ]:
# CONFIGURATION
DATA_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'# Enter dataset path
GENRES = ["blues", "classical", "country", "disco", "hiphop",
 "jazz", "metal", "pop", "reggae", "rock"] # Make the list of all genres available (alphabetical order)
STEMS = {'drums.wav','vocals.wav','bass.wav','other.wav'} # Write here stems file name
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00000'#Enter index as per Q10.


### Q1 & Q2

In [ ]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize empty dictionaries
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)

    # ------------------- write your code here -------------------------------

    # Counters
    corrupted_under_4kb = 0
    under_50491mb = 0
    over_50493mb = 0

    #  thresholds
    size_threshold_4kb = int(4 * 1024)  # 4 KB
    size_threshold_5mb_lower = int(5.0491 * 1024 * 1024)  # <5.0491 MB
    size_threshold_5mb_upper = int(5.0493 * 1024 * 1024)  # >5.0493 MB

    for g in GENRES:
        genre_path = os.path.join(root_dir, g)
        if not os.path.isdir(genre_path):
            continue

        valid_songs = []

        for song in os.listdir(genre_path):
            song_path = os.path.join(genre_path, song)
            if not os.path.isdir(song_path):
                continue

            is_valid = True
            for stem in STEMS:
                stem_path = os.path.join(song_path, stem)
                if not os.path.isfile(stem_path):
                    is_valid = False
                    break

                file_size = os.path.getsize(stem_path)

                # Count <4KB as corrupted
                if file_size < size_threshold_4kb:
                    corrupted_under_4kb += 1
                    is_valid = False
                # Count <5.0491MB
                elif file_size < size_threshold_5mb_lower:
                    under_50491mb += 1
                # Count >5.0493MB
                elif file_size > size_threshold_5mb_upper:
                    over_50493mb += 1
                
            if is_valid:
                valid_songs.append(song_path)
            

        # Stratified Shuffle Split
        rng.shuffle(valid_songs)
        split_idx = int(len(valid_songs) * (1 - val_split))
        train_songs = valid_songs[:split_idx]
        val_songs   = valid_songs[split_idx:]

        # Populate dictionaries
        for song_path in train_songs:
            for stem in STEMS:
                key = stem.replace('.wav', '')
                stem_path = os.path.join(song_path, stem)
                train_dataset[g][key].append(stem_path)

        for song_path in val_songs:
            for stem in STEMS:
                key = stem.replace('.wav', '')
                stem_path = os.path.join(song_path, stem)
                val_dataset[g][key].append(stem_path)

    # absolute difference 
    abs_diff = abs(over_50493mb - under_50491mb)

    # Print results
    print("Q1 Total <4KB:", corrupted_under_4kb)
    print("Q1 Total <5.0491MB:", under_50491mb)
    print("Q1 Final Answer (sum):", corrupted_under_4kb + under_50491mb)
    print("Q2 Absolute difference:", abs_diff)

    return train_dataset, val_dataset

# Run  function
tr, val = build_dataset(DATA_ROOT)


### Q3

In [ ]:
reggae_drums = len(tr['reggae']['drums'])
country_vocals = len(val['country']['vocals'])

diff = abs(reggae_drums - country_vocals)
print("Q3 Absolute difference:", diff)

## Q4

In [ ]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    """
    Input:
        dataset_dict: {genre: {stem: [paths...]}}
    Output:
        df: Pandas DataFrame containing details of all files with silence >= 5s
    """
    
    records = []
    total_files = 0

    for genre in dataset_dict:
        for stem_name in dataset_dict[genre]:
            for file_path in dataset_dict[genre][stem_name]:

                audio, _ = librosa.load(file_path, sr=sr)
                total_duration = len(audio) / sr

                # NON-SILENT INTERVALS
                intervals = librosa.effects.split(audio, top_db=top_db)

                silence_segments = []
                silence_type = []

                
                # Fully silent
                if len(intervals) == 0:
                    max_silence = total_duration
                    silence_type.append("Fully Silent")

                else:
                    max_silence = 0
                    prev_end = 0
                    # Start silence
                    first_start = intervals[0][0]
                    start_silence = first_start / sr
                    if start_silence > 0:
                        max_silence = max(max_silence, start_silence)
                        silence_type.append("Start Silence")

                    # Middle silence
                    for start, end in intervals:
                        middle_silence = (start - prev_end) / sr
                        if prev_end != 0 and middle_silence > 0:
                            max_silence = max(max_silence, middle_silence)
                            silence_type.append("Middle Silence")
                        prev_end = end

                    # End silence
                    end_silence = (len(audio) - prev_end) / sr
                    if end_silence > 0:
                        max_silence = max(max_silence, end_silence)
                        silence_type.append("End Silence")

                if max_silence >= threshold_sec:
                    total_files += 1
                    records.append({
                        "Genre": genre,
                        "Stem": stem_name,
                        "Duration": round(total_duration, 2),
                        "Max_Silence_Sec": round(max_silence, 2),
                        "Silence_Location": ", ".join(set(silence_type)),
                        "File_Path": file_path
                    })

    df = pd.DataFrame(records)
    return df

In [ ]:
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

print("Q4 Answer:", len(df_silence))

## Q5

In [ ]:
vocals_count = df_silence[df_silence["Stem"] == "vocals"].shape[0]
print("Q5 Answer:", vocals_count)

## Q6

In [ ]:
avg_vocals_silence = df_silence[df_silence["Stem"] == "vocals"]["Max_Silence_Sec"].mean()
print("Q6 Answer:", round(avg_vocals_silence, 2))

## Q7

In [ ]:
jazz_drums = df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums")
].shape[0]

print("Q7 Answer:", jazz_drums)

## Q8

In [ ]:
jazz_drums_middle_only = df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Silence_Location"] == "Middle Silence")
].shape[0]

print("Q8 Answer:", jazz_drums_middle_only)

## Q9

In [ ]:
jazz_drums_10sec = df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Max_Silence_Sec"] >= 10)
].shape[0]

print("Q9 Answer:", jazz_drums_10sec)

In [ ]:
def create_mix(song_path, sr=SR):
    stems_audio = []

    for stem in STEMS:
        stem_path = os.path.join(song_path, stem)
        audio, _ = librosa.load(stem_path, sr=sr)
        stems_audio.append(audio)

    # Make equal length 
    min_len = min(len(s) for s in stems_audio)
    stems_audio = [s[:min_len] for s in stems_audio]

    mix = np.sum(stems_audio, axis=0)

    return mix, sr

In [ ]:
mix, sr = create_mix(SONG_INDEX)

## Q10

In [ ]:
length_sec = len(mix) / sr
print("Q10 Length (sec):", round(length_sec, 2))

## Q11

In [ ]:
rms = np.sqrt(np.mean(mix**2))
print("Q11 RMS Amplitude:", round(rms, 4))

## Q12

In [ ]:
peak = np.max(np.abs(mix))
mix_normalized = mix / peak

max_value = np.max(np.abs(mix_normalized))
print("Q12 Max Peak Normalized Value:", max_value)